# Causal Discovery with py-tetrad

This notebook demonstrates a range of causal discovery algorithms available in [py-tetrad](https://github.com/cmu-phil/py-tetrad), a Python interface to the [Tetrad](https://github.com/cmu-phil/tetrad) causal search library.

**Citation:**
> Ramsey, J., & Andrews, B. (2023, November). Py-Tetrad and RPy-Tetrad: A New Python Interface with R Support for Tetrad Causal Search. In *Causal Analysis Workshop Series* (pp. 40–51). PMLR.

---

## Setup

Before running this notebook, make sure the following prerequisites are in place:

1. **Java 21+** must be installed. See [Setting up Java for Tetrad](https://github.com/cmu-phil/tetrad/wiki/Setting-up-Java-for-Tetrad).
2. **`JAVA_HOME`** must be set to your JDK path. Check with:
   ```
   echo $JAVA_HOME
   ```
3. **Python 3.5+** is required.
4. Install dependencies in your active virtual environment:
   ```
   pip install JPype1
   pip install git+https://github.com/cmu-phil/py-tetrad
   ```
5. The dataset used here (`airfoil-self-noise.continuous.txt`) is available in the `pytetrad/resources/` directory of the cloned repository.


## Imports and Data Loading

In [173]:
import pandas as pd
import pytetrad.tools.TetradSearch as ts

# Load the airfoil self-noise dataset (continuous variables)
data = pd.read_csv("resources/airfoil-self-noise.continuous.txt", sep="\t")
data = data.astype({col: "float64" for col in data.columns})

print(f"Dataset shape: {data.shape}")
data.head()

Dataset shape: (1503, 6)


,Frequency,Attack,Chord,Velocity,Displacement,Pressure
0,800.0,0.0,0.3048,71.3,0.0027,126.201
1,1000.0,0.0,0.3048,71.3,0.0027,125.201
2,1250.0,0.0,0.3048,71.3,0.0027,125.951
3,1600.0,0.0,0.3048,71.3,0.0027,127.591
4,2000.0,0.0,0.3048,71.3,0.0027,127.461


## Initialize TetradSearch

`TetradSearch` is a convenience wrapper around Tetrad's Java algorithms. It manages the score, independence test, knowledge constraints, and algorithm execution, hiding the JPype machinery.

In [174]:
search = ts.TetradSearch(data)
search.set_verbose(False)

## Score and Test

We use the **SEM BIC score** (linear Gaussian) and the **Fisher Z test** (Gaussian CI test) as the default score and test throughout. Background knowledge is also set here: we forbid the edge `Frequency → Attack`.

In [194]:
search.use_sem_bic()
search.use_fisher_z(alpha=0.01)

# Background knowledge: Frequency cannot cause Attack
search.set_forbidden("Frequency", "Attack")

---
## CPDAG Algorithms

The following algorithms return **CPDAGs** (Completed Partially Directed Acyclic Graphs), representing the Markov equivalence class of a DAG under the assumption of causal sufficiency (no latent confounders).

### FGES (Fast Greedy Equivalence Search)

A score-based algorithm that searches greedily over equivalence classes.

In [195]:
print("FGES")
search.run_fges()
print(search.get_graph_to_matrix())

FGES
here
   Frequency  Attack  Chord  Velocity  Displacement  Pressure
0          0       3      3         3             0         2
1          2       0      3         3             3         2
2          2       2      0         0             3         2
3          2       2      0         0             0         2
4          0       2      3         0             0         2
5          3       3      3         3             3         0


### BOSS (Best Order Score Search)

A permutation-based score algorithm that tends to outperform FGES on many benchmarks. Also demonstrates extracting a DAG from the CPDAG and the adjacency matrix.

In [196]:
print("BOSS")
search.run_boss(num_starts=1, use_bes=True, time_lag=0, use_data_order=True)
print(search.get_string())

# Extract one DAG from the CPDAG
dag = search.get_dag_java()
print("\nA DAG in the equivalence class:")
print(dag)

# Adjacency matrix
print("\nAdjacency matrix:")
print(search.get_graph_to_matrix())

BOSS
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --- Displacement
3. Attack --> Frequency
4. Displacement --> Chord
5. Displacement --> Pressure
6. Frequency --> Chord
7. Frequency --> Pressure
8. Pressure --> Chord
9. Velocity --> Chord
10. Velocity --> Frequency
11. Velocity --> Pressure

Graph Attributes:
Score: -46107.380370
BIC: -46026.912968

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882.65059720316;Chord: 2856.9619390766175;Velocity: -12518.376781557072;Displacement: 8815.0969551796;Pressure: -9054.27366892865]


A DAG in the equivalence class:
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --> Displacement
3. Attack --> Frequency
4. Displacement --> Chord
5. Displacement --> Pressure
6. Frequency --> Chord
7. Frequency --> Pressure
8. Pressure --> Chord
9. Velocity --> Chord
10. Velocity --> Frequency
11. Velocity --> Pr

### SP (Sparsest Permutation)

Searches for the sparsest permutation of variables; exact but slower than BOSS for larger graphs.

In [ ]:
print("SP")
search.run_sp()
print(search.get_string())

### GRaSP (Greedy Relaxations of Sparsest Permutation)

A scalable relaxation of SP.

In [ ]:
print("GRaSP")
search.run_grasp()
print(search.get_string())

### PC, PC-Max, and CPC

Constraint-based algorithms using the Fisher Z independence test. PC is the classic Peter-Clark algorithm; PC-Max and CPC are variants with different collider orientation strategies.

In [ ]:
print("PC")
search.run_pc()
print(search.get_string())

In [ ]:
print("PC-Max")
search.run_pc_max()
print(search.get_string())

In [ ]:
print("CPC (Conservative PC)")
search.run_cpc()
print(search.get_string())

---
## PAG Algorithms

The following algorithms return **PAGs** (Partial Ancestral Graphs), which represent Markov equivalence classes of MAGs and allow for the possibility of latent confounders and/or selection bias.

### FCI (Fast Causal Inference)

In [ ]:
print("FCI")
search.run_fci()
print(search.get_string())

### GFCI (Greedy FCI)

Combines FGES with FCI for improved performance.

In [ ]:
print("GFCI")
search.run_gfci()
print(search.get_string())

### FCIT (FCI with Targeted Testing)

FCIT is a hybrid PAG search algorithm that combines an FCI-style procedure with targeted conditional independence testing. It uses BOSS as a CPDAG oracle to guide the search, aiming to improve efficiency while still allowing for latent confounders.


In [ ]:
print("FCIT")
search.run_fcit()
print(search.get_string())

### GRaSP-FCI

In [ ]:
print("GRaSP-FCI")
search.run_grasp_fci()
print(search.get_string())

### BOSS-FCI

In [ ]:
print("BOSS-FCI")
search.run_boss_fci()
print(search.get_string())

### LV-Heuristic

In [ ]:
print("LV-Heuristic")
search.run_lv_heuristic()
print(search.get_string())

### CCD (Cyclic Causal Discovery)

Handles cyclic graphs; does not use background knowledge.

In [ ]:
print("CCD")
search.run_ccd()
print(search.get_string())

### SP-FCI

In [ ]:
print("SP-FCI")
search.run_sp_fci()
print(search.get_string())

---
## DAG Algorithms

The following algorithms return fully oriented **DAGs** under additional assumptions (e.g., non-Gaussianity or nonlinearity).

### LiNGAM (Linear Non-Gaussian Acyclic Model)

Exploits non-Gaussianity of noise to identify a unique DAG.

In [197]:
print("LiNGAM")
search.run_lingam(threshold_b=0.06)
print(search.get_string())
print("\nB-hat matrix (estimated coefficients):")
print(search.get_bhat())

LiNGAM
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Frequency
2. Chord --> Attack
3. Chord --> Frequency
4. Chord --> Pressure
5. Chord --> Velocity
6. Displacement --> Attack
7. Displacement --> Frequency
8. Displacement --> Pressure
9. Displacement --> Velocity
10. Pressure --> Attack
11. Pressure --> Frequency
12. Velocity --> Frequency

Graph Attributes:
BIC: -46145.149629

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882.65059720316;Chord: 2856.9619390766175;Velocity: -12518.376781557072;Displacement: 8815.0969551796;Pressure: -9054.27366892865]


B-hat matrix (estimated coefficients):
   Frequency     Attack        Chord   Velocity  Displacement    Pressure
0        0.0  91.252703 -3786.310697  31.377318 -86785.890321 -237.429938
1        0.0   0.000000   -33.928881   0.000000     88.663355   -0.327602
2        0.0   0.000000     0.000000   0.000000      0.000000    0.000000
3        0.0   0.000000     1.

### FASK (Fast Adjacency Skewness)

Uses skewness of residuals to orient edges.

In [198]:
print("FASK")
search.run_fask()
print(search.get_string())

FASK
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --> Frequency
3. Chord --> Displacement
4. Chord --> Pressure
5. Displacement --> Attack
6. Displacement --> Pressure
7. Frequency --> Velocity
8. Pressure --> Frequency
9. Velocity --> Pressure

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882.65059720316;Chord: 2856.9619390766175;Velocity: -12518.376781557072;Displacement: 8815.0969551796;Pressure: -9054.27366892865]



### LiNG-D (Linear Non-Gaussian with Dependence)

An extension of LiNGAM that handles cyclic models and returns potentially multiple stable B-hat matrices.

In [199]:
print("LiNG-D")
search.set_verbose(False)
search.run_lingd(threshold_w=1e-4)

print("\nUnstable B-hats:")
print(search.get_unstable_bhats())

print("\nStable B-hats:")
print(search.get_stable_bhats())

LiNG-D
To see unstable models and and their B matrices, set the verbose flag to true

Unstable B-hats:
LiNG-D Model #1 Stable = True

  V1          V2           V3        V4            V5         V6
      -1343.6234  -45577.5391   80.1209   119191.7371  -439.9922
                      -8.6212                795.6581           
                                                                
                       1.7148                  3.8879           
                                                                
          0.2163     -20.0525               -217.9665           

Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Frequency
2. Attack --> Pressure
3. Chord --> Attack
4. Chord --> Frequency
5. Chord --> Pressure
6. Chord --> Velocity
7. Displacement --> Attack
8. Displacement --> Frequency
9. Displacement --> Pressure
10. Displacement --> Velocity
11. Pressure --> Frequency
12. Velocity --> Frequency

Graph Node Attributes:
S

---
## Notes

- All algorithms above use the SEM BIC score and Fisher Z test set at the top of this notebook. You can substitute other scores and tests (e.g., `use_ebic()`, `use_kci()`, `use_ffml()`, `use_ffci()`) depending on your assumptions about the data-generating process.
- For **nonlinear data**, consider replacing `use_sem_bic()` with `use_ffml()` and `use_fisher_z()` with `use_ffci()`. See the [FFML/TRFF/FFCI paper](https://arxiv.org/abs/2605.05743) for details.
- For **time series data**, see the `run_svar_fci()` method and the `TsUtils` utilities in Tetrad.
- Full API documentation is available at the [Tetrad ReadTheDocs manual](https://tetrad-manual.readthedocs.io/en/latest/).